# 05 — Explicabilité : SHAP et LIME
**Projet** : ObRail MSPR 2025-2026  
**Auteure** : Charlotte  

**Objectif** : Expliquer les décisions du modèle LightGBM optimisé à deux niveaux :
- **SHAP** (SHapley Additive exPlanations) — importance globale des features sur l'ensemble du dataset
- **LIME** (Local Interpretable Model-agnostic Explanations) — explication de prédictions individuelles

**Pourquoi l'explicabilité est obligatoire pour ObRail ?**  
Le cahier des charges exige transparence et documentation des choix algorithmiques. Les décideurs européens et les ONG partenaires doivent pouvoir comprendre pourquoi une route est classifiée sous-desservie — une boîte noire n'est pas acceptable dans ce contexte réglementaire (cf. Ethics Guidelines for Trustworthy AI, Commission européenne).

**SHAP vs LIME — complémentarité** :
- SHAP : vision globale — quelles features comptent le plus sur l'ensemble du modèle ?
- LIME : vision locale — pourquoi *cette* route précise a-t-elle été flagguée ?

## 0. Imports et configuration

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer

SEED = 42
np.random.seed(SEED)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for folder in [cwd] + list(cwd.parents):
        if (folder / 'data').exists() and (folder / 'src').exists() and (folder / 'models').exists():
            return folder
    raise FileNotFoundError('Racine du projet introuvable.')

ROOT          = find_project_root()
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'models'
PLOT_DIR      = ROOT / 'evaluation' / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 120

print('✅ Imports OK')

## 1. Chargement du modèle et des données

In [ ]:
model = joblib.load(MODELS_DIR / 'best_model_optimized.joblib')

X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

feature_names = list(X_train.columns)

print(f'Modèle chargé : {type(model).__name__}')
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Features : {len(feature_names)}')

## 2. SHAP — Importance globale des features

**Comment fonctionne SHAP ?**  
SHAP attribue à chaque feature une valeur qui représente sa contribution à la prédiction, en moyenne sur toutes les combinaisons possibles de features (inspiré de la théorie des jeux de Shapley). Une valeur SHAP positive pousse la prédiction vers 1 (sous-desservie), une valeur négative vers 0.

Pour LightGBM on utilise `TreeExplainer` — optimisé pour les modèles à base d'arbres, beaucoup plus rapide que l'explainer générique.

In [ ]:
print('Calcul des valeurs SHAP sur X_test...')
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Pour la classification binaire LightGBM, shap_values peut être
# une liste [classe_0, classe_1] — on prend la classe 1 (sous-desservie)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f'✅ SHAP values calculées : shape {sv.shape}')

### 2.1 Importance moyenne des features (bar plot)

In [ ]:
# Importance = moyenne des valeurs absolues des SHAP values
shap_importance = pd.DataFrame({
    'feature':    feature_names,
    'importance': np.abs(sv).mean(axis=0),
}).sort_values('importance', ascending=False)

print('Top 15 features par importance SHAP :')
display(shap_importance.head(15))

# Visualisation top 15
top15 = shap_importance.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e74c3c' if 'log_distance' in f else '#3498db'
          for f in top15['feature']]
ax.barh(top15['feature'][::-1], top15['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Importance SHAP moyenne (|valeur|)')
ax.set_title('Top 15 features — Importance SHAP globale\n(rouge = log_distance, à surveiller)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shap_importance.png')
plt.show()

# Vérification du caveat log_distance
rank_log_dist = shap_importance['feature'].tolist().index('log_distance') + 1
print(f'\n→ log_distance est classée #{rank_log_dist} en importance SHAP')
if rank_log_dist == 1:
    print('  ⚠️  log_distance domine — le modèle apprend principalement la règle distance_km > 100')
    print('  Documenter cette limite dans le rapport.')
else:
    print('  ✅ log_distance ne domine pas — signal légitime confirmé')

### 2.2 SHAP Summary Plot (beeswarm)

Ce graphique montre pour chaque feature : la distribution des valeurs SHAP sur tous les exemples du test. La couleur indique la valeur de la feature (rouge = haute, bleu = basse).

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv,
    X_test,
    feature_names=feature_names,
    max_display=15,
    show=False,
)
plt.title('SHAP Summary Plot — impact des features sur is_underserved')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shap_summary.png', bbox_inches='tight')
plt.show()

### 2.3 SHAP Dependence Plot — log_distance

Montre comment la valeur SHAP de `log_distance` varie selon sa valeur. Si la relation est un saut net autour d'une valeur (correspondant à ~100km), cela confirme que le modèle a appris le seuil de la règle.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
shap.dependence_plot(
    'log_distance',
    sv,
    X_test,
    feature_names=feature_names,
    ax=ax,
    show=False,
)
# Ligne verticale à log(101) ≈ 4.62 — seuil de la règle distance_km > 100
ax.axvline(np.log1p(100), color='red', linestyle='--',
           label='Seuil règle : log(101) ≈ 4.62')
ax.legend()
ax.set_title('SHAP Dependence Plot — log_distance\n(ligne rouge = seuil distance_km > 100)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shap_dependence_log_distance.png')
plt.show()

print('Interprétation :')
print('Si les valeurs SHAP sautent nettement à la ligne rouge → le modèle a appris le seuil.')
print('Si la relation est progressive → le modèle capture une réalité opérationnelle continue.')

## 3. LIME — Explication de prédictions individuelles

**Comment fonctionne LIME ?**  
LIME génère des variations légères autour d'un exemple donné, observe comment les prédictions changent, et ajuste un modèle linéaire simple sur ces variations. Ce modèle linéaire local est facile à interpréter.

On explique trois cas concrets :
1. Une route correctement classée sous-desservie (vrai positif)
2. Une route correctement classée desservie (vrai négatif)
3. Un faux positif — route flagguée sous-desservie mais qui ne l'est pas

In [ ]:
# Initialisation de l'explainer LIME
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=['Non sous-desservi', 'Sous-desservi'],
    mode='classification',
    random_state=SEED,
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Identifier les cas
tp_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]
tn_idx = np.where((y_pred == 0) & (y_test.values == 0))[0]
fp_idx = np.where((y_pred == 1) & (y_test.values == 0))[0]

print(f'Vrais positifs  (TP) disponibles : {len(tp_idx)}')
print(f'Vrais négatifs  (TN) disponibles : {len(tn_idx)}')
print(f'Faux positifs   (FP) disponibles : {len(fp_idx)}')

In [ ]:
def explain_and_save(idx, label, filename):
    """Génère et sauvegarde une explication LIME pour un exemple donné."""
    instance = X_test.iloc[idx].values
    proba    = y_proba[idx]
    true_val = y_test.iloc[idx]
    pred_val = y_pred[idx]

    print(f'\n{"="*55}')
    print(f'Cas : {label}')
    print(f'  Vraie valeur  : {int(true_val)} ({"sous-desservie" if true_val == 1 else "desservie"})')
    print(f'  Prédiction    : {int(pred_val)} ({"sous-desservie" if pred_val == 1 else "desservie"})')
    print(f'  Probabilité   : {proba:.4f}')

    exp = lime_explainer.explain_instance(
        instance,
        model.predict_proba,
        num_features=10,
        num_samples=500,
    )

    print(f'\nTop features LIME (impact sur prédiction "Sous-desservi") :')
    for feat, weight in exp.as_list(label=1):
        direction = '↑' if weight > 0 else '↓'
        print(f'  {direction} {feat} : {weight:.4f}')

    # Sauvegarde du graphique
    fig = exp.as_pyplot_figure(label=1)
    fig.suptitle(f'LIME — {label}\n(p={proba:.3f}, vrai={int(true_val)}, prédit={int(pred_val)})',
                 fontsize=10)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, bbox_inches='tight')
    plt.show()
    plt.close()

    return exp

print('✅ Fonction LIME définie')

### 3.1 Cas 1 — Vrai positif (route correctement identifiée comme sous-desservie)

In [ ]:
exp_tp = explain_and_save(
    idx=tp_idx[0],
    label='Vrai positif — route sous-desservie correctement détectée',
    filename='lime_true_positive.png',
)

### 3.2 Cas 2 — Vrai négatif (route correctement identifiée comme desservie)

In [ ]:
exp_tn = explain_and_save(
    idx=tn_idx[0],
    label='Vrai négatif — route desservie correctement identifiée',
    filename='lime_true_negative.png',
)

### 3.3 Cas 3 — Faux positif (route flagguée sous-desservie à tort)

Ce cas est particulièrement intéressant pour ObRail — comprendre pourquoi le modèle se trompe permet d'identifier ses limites et d'améliorer les features futures.

In [ ]:
exp_fp = explain_and_save(
    idx=fp_idx[0],
    label='Faux positif — route desservie flagguée sous-desservie à tort',
    filename='lime_false_positive.png',
)

### 3.4 Sauvegarde de l'explication LIME principale

In [ ]:
# Sauvegarde de l'explication du vrai positif comme livrable principal
# (fichier attendu par le cahier des charges : lime_explanation.png)
fig = exp_tp.as_pyplot_figure(label=1)
fig.suptitle('LIME — Explication principale (vrai positif)', fontsize=10)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'lime_explanation.png', bbox_inches='tight')
plt.close()
print('✅ Sauvegardé → evaluation/plots/lime_explanation.png')

## 4. Synthèse et conclusions pour le rapport

**Ce que SHAP nous apprend**  
- Les features les plus importantes sont identifiées et classées
- Le dependence plot de `log_distance` permet de vérifier si le modèle apprend le seuil de la règle ou une réalité opérationnelle continue
- Si `log_distance` domine, c'est une limite à documenter dans le rapport

**Ce que LIME nous apprend**  
- Pour chaque route individuelle, LIME explique quelles features ont poussé la prédiction dans un sens ou dans l'autre
- Le cas faux positif est le plus instructif : il montre les situations où le modèle se trompe et pourquoi

**Lien avec les exigences ObRail**  
- Transparence : les décisions du modèle sont explicables à un public non technique
- RGPD / Ethics Guidelines : le modèle n'est pas une boîte noire — ses décisions sont auditables
- Amélioration future : les explications LIME sur les faux positifs orientent le travail de feature engineering futur

**Plots produits** :
- `evaluation/plots/shap_importance.png` — importance globale des features
- `evaluation/plots/shap_summary.png` — beeswarm plot
- `evaluation/plots/shap_dependence_log_distance.png` — vérification caveat log_distance
- `evaluation/plots/lime_explanation.png` — explication principale (livrable cahier des charges)
- `evaluation/plots/lime_true_positive.png` — vrai positif
- `evaluation/plots/lime_true_negative.png` — vrai négatif
- `evaluation/plots/lime_false_positive.png` — faux positif